# edge-tts voice playground

Pick a narrator from a dropdown, type some text, hear it, save the MP3.

The dropdown is populated from `edge_tts.list_voices()` — the same data behind
`edge-tts --list-voices`, but as structured dicts instead of text you'd have to grep.

## 1. Imports and a small async helper

Jupyter already has an event loop running, so `asyncio.run()` blows up if called
directly. Running the coroutine in a worker thread — which has no loop of its own —
sidesteps that without needing `nest_asyncio`.

In [ ]:
import asyncio
import base64
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import edge_tts
import ipywidgets as widgets
from IPython.display import Audio, HTML, display

_pool = ThreadPoolExecutor(max_workers=1)


def run_async(coro_factory):
    """Run an async function to completion from inside a notebook cell or callback."""
    return _pool.submit(lambda: asyncio.run(coro_factory())).result()


OUTPUT_DIR = Path("audio")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Fetch the voice list

Change `LOCALE_FILTER` to widen the net — `"en-"` gets every English variant,
`""` gets all ~320 voices across every language.

In [ ]:
LOCALE_FILTER = "en-US"

all_voices = run_async(edge_tts.list_voices)
voices = sorted(
    (v for v in all_voices if v["Locale"].startswith(LOCALE_FILTER)),
    key=lambda v: v["ShortName"],
)

print(f"{len(voices)} voices matching {LOCALE_FILTER!r} "
      f"(out of {len(all_voices)} total)\n")

for v in voices:
    tags = v.get("VoiceTag", {}) or {}
    personality = ", ".join(tags.get("VoicePersonalities", [])) or "—"
    print(f"{v['ShortName']:<34} {v['Gender']:<7} {personality}")

## 3. Build the dropdown labels

`ShortName` is what the synthesiser wants, but it's not much to choose from.
Pairing it with the personality tags Microsoft ships makes the dropdown actually
useful for picking a narrator.

In [ ]:
def label_for(v):
    tags = v.get("VoiceTag", {}) or {}
    personality = ", ".join(tags.get("VoicePersonalities", [])[:3])
    name = v["ShortName"].split("-")[-1].replace("Neural", "")
    bits = [name, v["Gender"]]
    if personality:
        bits.append(personality)
    return f"{' · '.join(bits)}  ({v['ShortName']})"


voice_options = [(label_for(v), v["ShortName"]) for v in voices]
default_voice = next(
    (sn for _, sn in voice_options if sn == "en-US-AvaNeural"),
    voice_options[0][1],
)

len(voice_options), default_voice

## 4. The widget

Rate and pitch are sent to the service as signed strings (`-10%`, `+5Hz`), which is
why the sliders get formatted rather than passed as raw numbers.

In [ ]:
voice_dd = widgets.Dropdown(
    options=voice_options,
    value=default_voice,
    description="Voice:",
    layout=widgets.Layout(width="640px"),
    style={"description_width": "70px"},
)

text_area = widgets.Textarea(
    value="The quick brown fox jumps over the lazy dog. "
          "Numbers like 1,234 and dates like March 3rd are handled too.",
    description="Text:",
    layout=widgets.Layout(width="640px", height="120px"),
    style={"description_width": "70px"},
)

rate_slider = widgets.IntSlider(
    value=0, min=-50, max=50, step=5, description="Rate %:",
    continuous_update=False, style={"description_width": "70px"},
)

pitch_slider = widgets.IntSlider(
    value=0, min=-50, max=50, step=5, description="Pitch Hz:",
    continuous_update=False, style={"description_width": "70px"},
)

save_box = widgets.Checkbox(value=False, description="Keep the MP3/SRT in ./audio/")
generate_btn = widgets.Button(description="Generate", button_style="primary", icon="play")
out = widgets.Output()


def slugify(voice, text):
    stem = re.sub(r"[^a-z0-9]+", "-", text.lower())[:32].strip("-") or "clip"
    return f"{voice}--{stem}.mp3"


def download_link(path, label):
    """Build a self-contained <a download> link so the file survives cell re-runs
    even if it's deleted from disk right after (base64-embedded, no server needed)."""
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f'<a download="{path.name}" href="data:text/plain;base64,{data}">{label}</a>'


def synthesize(text, voice, rate, pitch, audio_path, srt_path=None):
    async def _run():
        communicate = edge_tts.Communicate(
            text, voice, rate=f"{rate:+d}%", pitch=f"{pitch:+d}Hz"
        )
        if srt_path is None:
            await communicate.save(str(audio_path))
            return

        submaker = edge_tts.SubMaker()
        with open(audio_path, "wb") as f:
            async for chunk in communicate.stream():
                if chunk["type"] == "audio":
                    f.write(chunk["data"])
                elif chunk["type"] in ("WordBoundary", "SentenceBoundary"):
                    submaker.feed(chunk)
        srt_path.write_text(submaker.get_srt(), encoding="utf-8")

    run_async(_run)
    return audio_path


def on_generate(_):
    with out:
        out.clear_output(wait=True)
        text = text_area.value.strip()
        if not text:
            print("Nothing to say.")
            return

        voice = voice_dd.value
        path = OUTPUT_DIR / slugify(voice, text)
        srt_path = path.with_suffix(".srt")
        print(f"Synthesising with {voice} …")

        try:
            synthesize(text, voice, rate_slider.value, pitch_slider.value, path, srt_path)
        except Exception as exc:
            out.clear_output(wait=True)
            print(f"Failed: {type(exc).__name__}: {exc}")
            print("If this is a connection or 403 error, the Edge endpoint may have "
                  "shifted — try `uv lock --upgrade-package edge-tts`.")
            return

        out.clear_output(wait=True)
        size_kb = path.stat().st_size / 1024
        print(f"{voice}  ·  {size_kb:.0f} KB")
        display(Audio(str(path), autoplay=True))
        display(HTML(download_link(srt_path, "⬇ Download subtitles (.srt)")))

        if not save_box.value:
            path.unlink(missing_ok=True)
            srt_path.unlink(missing_ok=True)
        else:
            print(f"Saved to {path} and {srt_path}")


generate_btn.on_click(on_generate)

display(widgets.VBox([
    voice_dd,
    text_area,
    widgets.HBox([rate_slider, pitch_slider]),
    widgets.HBox([generate_btn, save_box]),
    out,
]))